In [ ]:
# ============================================
# CELL 1: Setup & Load Dataset
# ============================================
from pathlib import Path
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score

try:
    PROJECT_ROOT = Path(__file__).parent.parent
except NameError:
    cwd = Path.cwd()
    PROJECT_ROOT = cwd.parent if cwd.name == "notebooks" else cwd

data_processed = PROJECT_ROOT / "data" / "processed"

df = pd.read_parquet(data_processed / "makassar_final_dataset.parquet")

print("DATA LOADED:", df.shape)


# ============================================
# CELL 2: Temporal Train-Test Split (IMPORTANT)
# ============================================
df["date"] = pd.to_datetime(df["date"])

train_df = df[df["date"] < "2021-01-01"]
test_df  = df[df["date"] >= "2021-01-01"]

print("\n=== SPLIT INFO ===")
print("TRAIN:", train_df.shape)
print("TEST :", test_df.shape)


# ============================================
# CELL 3: FEATURE SETS (FOR SHORTCUT DIAGNOSIS)
# ============================================

# ONLY SPATIAL (shortcut test)
spatial_features = ["grid_id"]

# ONLY TEMPORAL (physics signal)
temporal_features = [
    "rain_real_lag_1",
    "rain_real_lag_3",
    "rain_real_lag_7",
    "rain_real_lag_14",
    "rain_real_7d_sum",
    "rain_real_30d_sum",
]

# FULL MODEL
full_features = spatial_features + temporal_features + ["spatial_factor"]


# ============================================
# CELL 4: DATA PREP
# ============================================
y_train = train_df["flood_label_v2"]
y_test  = test_df["flood_label_v2"]


# ============================================
# CELL 5: MODEL 1 — SPATIAL ONLY (SHORTCUT TEST)
# ============================================
X_train_spatial = train_df[spatial_features]
X_test_spatial  = test_df[spatial_features]

model_spatial = RandomForestClassifier(
    n_estimators=80,
    n_jobs=-1,
    random_state=42
)

model_spatial.fit(X_train_spatial, y_train)
pred_spatial = model_spatial.predict_proba(X_test_spatial)[:, 1]

auc_spatial = roc_auc_score(y_test, pred_spatial)

print("\n=== MODEL 1: SPATIAL ONLY ===")
print("AUC:", auc_spatial)


# ============================================
# CELL 6: MODEL 2 — TEMPORAL ONLY (PHYSICS SIGNAL)
# ============================================
X_train_temp = train_df[temporal_features]
X_test_temp  = test_df[temporal_features]

model_temp = RandomForestClassifier(
    n_estimators=120,
    n_jobs=-1,
    random_state=42
)

model_temp.fit(X_train_temp, y_train)
pred_temp = model_temp.predict_proba(X_test_temp)[:, 1]

auc_temp = roc_auc_score(y_test, pred_temp)

print("\n=== MODEL 2: TEMPORAL ONLY ===")
print("AUC:", auc_temp)


# ============================================
# CELL 7: MODEL 3 — FULL MODEL
# ============================================
X_train_full = train_df[full_features]
X_test_full  = test_df[full_features]

model_full = RandomForestClassifier(
    n_estimators=150,
    n_jobs=-1,
    random_state=42
)

model_full.fit(X_train_full, y_train)
pred_full = model_full.predict_proba(X_test_full)[:, 1]

auc_full = roc_auc_score(y_test, pred_full)

print("\n=== MODEL 3: FULL MODEL ===")
print("AUC:", auc_full)


# ============================================
# CELL 8: SHORTCUT LEARNING SCORE (CORE Q1 METRIC)
# ============================================

shortcut_score = (auc_full - auc_temp) / auc_full

print("\n=== SHORTCUT LEARNING SCORE ===")
print("S =", shortcut_score)


# ============================================
# CELL 9: FINAL INTERPRETATION
# ============================================

print("\n=== FINAL RESULT ===")
print("AUC Spatial :", auc_spatial)
print("AUC Temporal :", auc_temp)
print("AUC Full     :", auc_full)

if auc_spatial > 0.6:
    print("\n⚠ WARNING: Spatial leakage detected")
else:
    print("\n✔ Spatial shortcut NOT dominant")

if auc_temp > 0.9:
    print("✔ Strong physical signal detected")
else:
    print("⚠ Weak temporal physics signal")

if shortcut_score > 0.1:
    print("⚠ Model still relies on shortcuts")
else:
    print("✔ Good generalization behavior")

DATA LOADED: (4922658, 12)

=== SPLIT INFO ===
TRAIN: (2853838, 12)
TEST : (2068820, 12)

=== MODEL 1: SPATIAL ONLY ===
AUC: 0.5864705279672076

=== MODEL 2: TEMPORAL ONLY ===
AUC: 0.6844848792415915
